# Latency vs Throughput

Reads the cross-experiment ledger `results/summary_ledger.csv` produced by the
benchmark controller and plots the latency-throughput frontier (output TPS vs
p95 latency) and TTFT vs concurrency.

First produce results (no GPU needed):
`python -m benchmarks.runner.client --config benchmarks/configs/concurrency_sweep.yaml --backend mock --out results`

Requires analysis extras: `pip install -e ".[analysis]"`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

ledger = Path("../../results/summary_ledger.csv")
assert ledger.exists(), f"Run benchmarks first; missing {ledger}"
df = pd.read_csv(ledger)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for backend, g in df.groupby("backend"):
    gg = g.sort_values("output_tps")
    axes[0].plot(gg["output_tps"], gg["latency_p95_s"], "o-", label=backend)
    gc = g.sort_values("concurrency")
    axes[1].plot(gc["concurrency"], gc["ttft_p50_s"] * 1000, "o-", label=backend)
axes[0].set(xlabel="Output TPS", ylabel="p95 latency (s)", title="Latency vs Throughput")
axes[1].set(xlabel="Concurrency", ylabel="TTFT p50 (ms)", title="TTFT vs Concurrency")
for ax in axes:
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
df